In [21]:
import pandas as pd
import numpy as np

In [22]:
PATH = './data'
TARGET = 'SalePrice'

In [23]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

In [24]:
df = pd.read_csv(PATH + "/train.csv")

In [53]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, shuffle=True, random_state=42
)

train_ids = X_train["Id"]
test_ids = X_test["Id"]

print(f"X_train shape {X_train.shape}")
print(f"X_test shape {X_test.shape}")

X_train shape (1168, 80)
X_test shape (292, 80)


### Data Cleaning ###

### Ordinal Columns ###

In [54]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

ord_categories = [
    exter_qual_order,
    exter_cond_order,
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Categorical One-Hot-Encoded Columns ###

In [55]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

### Numeric One-Hot-Encoded Columns ###

In [56]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Numeric Imputing ###

In [57]:
num_nan_cols = [
    'LotFrontage',
    'GarageYrBlt',
    'MasVnrArea',
]

### Data Cleanup Pipeline ###

In [58]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
    ('ordinal', OrdinalEncoder(categories=ord_categories)),
])

num_ohe_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('ord',     ordinal_pipeline,               cat_ord_cols),
        ('cat_ohe', ohe_pipeline,                   ohe_cols),
        ('num_ohe', num_ohe_pipeline,               num_ohe_cols),
        ('num_nan', SimpleImputer(strategy='mean'), num_nan_cols),
        ('drop',    'drop',                         ['Id'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('cat_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featur

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFE

class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=.5):
        pass

    def fit(self, X, y):
        pass

    def transform(self, X):
        pass

In [131]:
from sklearn.feature_selection import RFE

preprocessor_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    # ('rfe', RFE(estimator=LinearRegression(), n_features_to_select=10))
])

In [132]:
preprocessor_pipeline.fit_transform(X_train, y_train)
X_train_t = preprocessor_pipeline.transform(X_train)
X_test_t = preprocessor_pipeline.transform(X_test)
y_train_t = np.log1p(y_train)
y_test_t = np.log1p(y_test)

print(X_train_t.shape)
print(X_test_t.shape)
print(y_train_t.shape)
print(y_test_t.shape)

(1168, 246)
(292, 246)
(1168,)
(292,)


### Full Pipeline ###

In [142]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge(alpha=10))
])

param_grid = {
    'model__alpha': [.01, .05, .1, .5, 1, 5, 10, 15, 20, 50, 100]
}

search = GridSearchCV(
    final_pipeline,
    param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

search.fit(X_train, y_train_t)
cv_results_df = pd.DataFrame(search.cv_results_)

cv_results_df

# cv_result = cross_validate(
#     final_pipeline, X_train, y_train_t,
#     cv=5,
#     scoring='neg_root_mean_squared_error',
#     return_train_score=True
# )
#
# cv_result_df = pd.DataFrame(cv_result)
#
# train_score = -cv_result_df['train_score'].mean()
# test_score  = -cv_result_df['test_score'].mean()
#
# print(f'train_score {train_score}')
# print(f'test_score {test_score}')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__alpha,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.085831,0.030899,0.048609,0.018507,0.01,{'model__alpha': 0.01},-0.137029,-0.160949,-0.207481,-0.131599,-0.159534,-0.159318,0.026786,19,-0.087785,-0.087278,-0.091634,-0.092654,-0.088958,-0.089662,0.002123
1,0.075586,0.002028,0.046646,0.001273,0.05,{'model__alpha': 0.05},-0.136852,-0.157164,-0.205551,-0.129066,-0.153021,-0.156331,0.026672,18,-0.087987,-0.087525,-0.091685,-0.092878,-0.089275,-0.089870,0.002085
2,0.074091,0.001457,0.047561,0.000469,0.10,{'model__alpha': 0.1},-0.136855,-0.153899,-0.204642,-0.127133,-0.148172,-0.154140,0.026886,17,-0.088441,-0.088080,-0.091743,-0.093373,-0.089934,-0.090314,0.002002
3,0.074864,0.001314,0.047725,0.001096,0.50,{'model__alpha': 0.5},-0.137944,-0.144412,-0.201339,-0.122162,-0.133625,-0.147896,0.027688,16,-0.092576,-0.093134,-0.092425,-0.097828,-0.095616,-0.094316,0.002099
4,0.089064,0.017716,0.048072,0.001509,1.00,{'model__alpha': 1},-0.138482,-0.141257,-0.199323,-0.120490,-0.127799,-0.145470,0.027939,15,-0.095869,-0.096975,-0.093351,-0.101348,-0.099936,-0.097496,0.002860
5,0.079408,0.013335,0.047048,0.001495,5.00,{'model__alpha': 5},-0.137492,-0.137426,-0.196043,-0.117567,-0.117563,-0.141218,0.028820,12,-0.104325,-0.105883,-0.097503,-0.110082,-0.109860,-0.105531,0.004593
6,0.072845,0.000897,0.046598,0.000681,10.00,{'model__alpha': 10},-0.136339,-0.137665,-0.196298,-0.116956,-0.114193,-0.140290,0.029614,11,-0.108024,-0.109339,-0.100026,-0.113643,-0.113712,-0.108949,0.005007
7,0.072621,0.000279,0.046949,0.000404,11.00,{'model__alpha': 11},-0.136173,-0.137803,-0.196442,-0.116917,-0.113780,-0.140223,0.029750,10,-0.108554,-0.109811,-0.100414,-0.114138,-0.114249,-0.109433,0.005051
8,0.078884,0.013964,0.047162,0.000830,12.00,{'model__alpha': 12},-0.136024,-0.137951,-0.196597,-0.116892,-0.113415,-0.140176,0.029879,8,-0.109045,-0.110243,-0.100777,-0.114593,-0.114743,-0.109880,0.005090
9,0.079464,0.014860,0.046304,0.000712,13.00,{'model__alpha': 13},-0.135889,-0.138106,-0.196760,-0.116877,-0.113089,-0.140144,0.030003,6,-0.109503,-0.110643,-0.101120,-0.115016,-0.115202,-0.110297,0.005124
